In [94]:
import pandas as pd   
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder
import pickle

In [95]:
data=pd.read_csv("Churn_Modelling.csv")
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [96]:
data = data.drop(['RowNumber','CustomerId', 'Surname'],axis=1,errors='ignore')
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [97]:
#works on the gender categroy to convert it into numeric
label_encoder_gender=LabelEncoder()
#creates and object that can convert data into integers
#values so machine learning algorithms can work with them.
data['Gender']=label_encoder_gender.fit_transform(data['Gender'])
#fit learns unique categories and transform converts
#into number-----fit transfrom does both together
data                                                                      

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,1,39,5,0.00,2,1,0,96270.64,0
9996,516,France,1,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,0,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,1,42,3,75075.31,2,1,0,92888.52,1


In [98]:
from sklearn.preprocessing import OneHotEncoder#(categories to binary columns) cant work with labelEncoder due to more than 2values
one_hot_encoder_geo=OneHotEncoder()#create one hot encoder object
geo_encoder=one_hot_encoder_geo.fit_transform(data[['Geography']])
geo_encoder


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 10000 stored elements and shape (10000, 3)>

In [99]:
geo_encoder.toarray()#fit transform returns a sparse matrix so we need toarray() to view actual values
#sparse matrix view only non zero elements but now we use to array to view dense matrix

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]], shape=(10000, 3))

In [100]:
one_hot_encoder_geo.get_feature_names_out(['Geography'])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [101]:
geo_encoder_df = pd.DataFrame(
    geo_encoder.toarray(),
    columns=one_hot_encoder_geo.get_feature_names_out(['Geography'])
)
#.toarray() converts the sparse matrix into a full normal matrix with zeros, while
#get_feature_names_out() provides names for the encoded columns.

In [102]:
##combine one hot encoder table with the original data
data=pd.concat([data.drop('Geography',axis=1),geo_encoded_df],axis=1) #axis1==column operation
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [103]:
# save the encoder and scalar
with open('label_encoder_gender.pkl','wb') as file:#create a file(wb=write binary mode)
    pickle.dump(label_encoder_gender,file)#Writes the label_encoder_gender into the file
#pickle.dump serializes a Python object (any object — model, encoder, scaler) into bytes and writes it to a file
with open('one_hot_encoder_geo.pkl','wb') as file:
    pickle.dump(one_hot_encoder_geo,file)

# freezing the trained encoders into files later to be used in deployment

In [104]:
# divide the dataset into dependent and independent features
X=data.drop('Exited',axis=1)# all columns except Exited---everything the modle learns from
y=data['Exited'] #only the column Exited---what model is trying to learn

#split the data set into training and testing
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)#0.2=80%training 20 testing

#scale these features
Scaler=StandardScaler()# normalizes already numeric data
X_train=Scaler.fit_transform(X_train)
X_test=Scaler.transform(X_test)
#The model trains on X_train, then gets evaluated on X_test which it has never seen

In [105]:
#Train once → fit scaler → save scaler → never touch it again

In [106]:
with open('scaler.pkl','wb') as file:#saving the scaler
    pickle.dump(Scaler,file)

In [107]:
import numpy as np
print(np.__version__)  # must be 1.x


2.2.6


In [108]:
import tensorflow as tf #machine learning/deep learning framework used to build and train models.
from tensorflow.keras.models import Sequential #--helps you stack layers in straight line
from tensorflow.keras.layers import Dense #creates fully connected layer
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard #EarlyStopping=stops if no improvements,
#Tensorboard---lets you see graph
import datetime

In [109]:
X_train.shape[1]#no of columns in table----this is the input layer

12

In [110]:
#Build our ANN model
#keras sequential neural network---  builds a feedforward neural network with 3 layers:
from keras.layers import Input
model=Sequential([
    Input(shape=(X_train.shape[1],)),  #input layer
    Dense(64,activation='relu'),#HL1 
    Dense(32,activation='relu'),##HL2
    Dense(1,activation='sigmoid')#output layer___single neuron outputting value between 0 and 1
#sigmoid squashes output to a probability
])#funnel shaped

In [111]:
model.summary()

#dense64= (8neuron*64neuron) + 64bias=576
#dense32= (64*32)+32=2080
#dense1= (32*1)+1=33

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_9 (Dense)                 │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [112]:
import tensorflow
opt=tensorflow.keras.optimizers.Adam(learning_rate=0.01)
loss=tensorflow.keras.losses.BinaryCrossentropy()
loss

<LossFunctionWrapper(<function binary_crossentropy at 0x1453f45e0>, kwargs={'from_logits': False, 'label_smoothing': 0.0, 'axis': -1})>

In [113]:
#compile the model
model.compile(
    optimizer='adam',# how to update weight
    loss='binary_crossentropy',#how to measure error
    metrics=['accuracy']#how to evaluate perfromance
    )

In [114]:
#setup the Tensorboard
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
log_dir='log/fit'+datetime.datetime.now().strftime(("%Y%m%d-%H%M%S"))
tensorflow_callback=TensorBoard(log_dir=log_dir,histogram_freq=1)

In [115]:
#Set up Early Stopping
early_stopping = EarlyStopping(
    monitor='val_loss',   # watch validation loss
    patience=10,          # stop if no improvement for 10 epochs
    restore_best_weights=True  # roll back to the best epoch
)

In [116]:
#train the model
history=model.fit(
    X_train, #input data
    y_train, #target outcomes    #X_train → predict → y_train
    validation_data=(X_test,y_test), #check how well the model generalizes to new data.
    epochs=100,
    callbacks=[tensorflow_callback,early_stopping_callback]   
)

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.8012 - loss: 0.4606 - val_accuracy: 0.8270 - val_loss: 0.3915
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 796us/step - accuracy: 0.8440 - loss: 0.3793 - val_accuracy: 0.8530 - val_loss: 0.3524
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 798us/step - accuracy: 0.8555 - loss: 0.3531 - val_accuracy: 0.8575 - val_loss: 0.3449
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 783us/step - accuracy: 0.8590 - loss: 0.3445 - val_accuracy: 0.8550 - val_loss: 0.3459
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 790us/step - accuracy: 0.8605 - loss: 0.3399 - val_accuracy: 0.8605 - val_loss: 0.3430
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 798us/step - accuracy: 0.8625 - loss: 0.3368 - val_accuracy: 0.8570 - val_loss: 0.3407
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 806us/step - accuracy: 0.8633 - loss: 0.3329 - val_accuracy: 0.8605 - val_loss: 0.3441
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 794us/step - accuracy: 0.8648 - loss: 0.3

In [117]:
model.save('model.h5')

In [118]:
#load tensorboard extension
%load_ext tensorboard


The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [119]:

print(log_dir)


log/fit20260525-215211


Reusing TensorBoard on port 6008 (pid 11996), started 1 day, 3:07:17 ago. (Use '!kill 11996' to kill it.)

In [93]:
## load the pickle file
